# 🚀 AIC 2026: SigLIP-SO400M Multi-GPU Feature Extraction & Indexing Pipeline

This notebook is dedicated exclusively to **high-resolution SigLIP-SO400M visual feature extraction**:
1. **Vision Backbone**: Google `siglip-so400m-patch14-384` (1152-dimensional L2-normalized vectors in FP16).
2. **Adaptive Hybrid Keyframe Decoding**: PyAV C-level I-frame decoding + grab-stride gap filling (<= 2.5s) + talking-head pruning.
3. **Aspect-Ratio Preserving Preprocessing**: Bicubic antialiasing (`cv2.INTER_CUBIC`) with neutral letterbox padding (avoids 16:9 to 1:1 squishing).
4. **Multi-Process CPU Prefetching**: 3-4 background worker processes per GPU keep Tensor Cores 100% saturated.
5. **Multi-GPU Parallel Sharding**: Automatically detects and distributes video files across all available GPUs (e.g. 2x T4 on Kaggle).
6. **Index Generation & Packaging**: Streams extracted features directly into FAISS FlatIP index and outputs a single `.zip` bundle.

In [ ]:
# 1. Clone or Pull Latest Repository
import os, sys

!git clone https://github.com/TotallyNotMinh/aic2026.git || (cd aic2026 && git pull)

REPO_DIR = "/kaggle/working/aic2026"
if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f"[*] Active working directory: {os.getcwd()}")

In [ ]:
# 2. Verify GPU Hardware & CUDA Availability
import torch

num_gpus = torch.cuda.device_count()
print(f"[*] Detected {num_gpus} CUDA GPU(s):")
for i in range(num_gpus):
    props = torch.cuda.get_device_properties(i)
    print(f"    • GPU {i}: {props.name} | VRAM: {props.total_memory / 1e9:.2f} GB")

if num_gpus == 0:
    print("[!] Warning: No GPU detected. Running on CPU will be significantly slower.")

In [ ]:
# 3. Install Minimal Dependencies
!pip install -q av transformers torch torchvision opencv-python-headless tqdm faiss-cpu
!pip install -q "pillow<11.0.0"

In [ ]:
# 4. Link Kaggle Input Video Folders into data/
import os, glob

target_dir = os.path.join(os.getcwd(), "data")
os.makedirs(target_dir, exist_ok=True)

if os.path.exists("/kaggle/input"):
    print("[*] Linking video folders from /kaggle/input into data/...")
    for p in glob.glob("/kaggle/input/**/Videos_*", recursive=True):
        dest = os.path.join(target_dir, os.path.basename(p))
        if not os.path.exists(dest):
            try:
                os.symlink(p, dest)
            except Exception:
                pass

available_vids = glob.glob(os.path.join(target_dir, "Videos_*", "video", "*.mp4"))
print(f"[✓] Found {len(available_vids)} MP4 video files ready for extraction.")

In [ ]:
# 5. Run Parallel Multi-GPU SigLIP-SO400M Extraction
!bash scripts/run_dual_gpu.sh


In [ ]:
# 6. Verify Extracted Feature Dimensions & Metadata Quality
import glob, json, numpy as np

npy_files = sorted(glob.glob("cache/siglip_features/*.npy"))
meta_files = sorted(glob.glob("cache/siglip_meta/*.json"))

print(f"[*] Total Extracted Feature Files: {len(npy_files)}")
print(f"[*] Total Metadata Files: {len(meta_files)}")

if npy_files:
    sample_vec = np.load(npy_files[0])
    print(f"[*] Sample Matrix Shape: {sample_vec.shape} (Expected: N x 1152)")
    print(f"[*] Sample Vector L2 Norm: {np.linalg.norm(sample_vec[0]):.4f} (Expected: 1.0000)")
    with open(meta_files[0], "r", encoding="utf-8") as f:
        sample_meta = json.load(f)
    print(f"[*] Sample Keyframe Metadata:", sample_meta[0])

In [ ]:
# 7. Build Production FAISS FlatIP & Unified Metadata Index
from scripts.build_faiss_index import build_production_faiss_index

print("[*] Compiling production FAISS index from extracted SigLIP features...")
build_production_faiss_index(
    siglip_dir="cache/siglip_features",
    siglip_meta_dir="cache/siglip_meta",
    output_prefix="cache/faiss_siglip"
)
print("[✓] FAISS FlatIP Index built successfully at cache/faiss_siglip.index")

In [ ]:
# 8. Package & Create Direct Download Link for SigLIP Embeddings
import os, zipfile
from IPython.display import FileLink, display

zip_filename = "/kaggle/working/siglip_features_aic2026.zip"
if not os.path.exists(os.path.dirname(zip_filename)):
    zip_filename = "siglip_features_aic2026.zip"

print(f"[*] Compressing all SigLIP features & FAISS index into {zip_filename}...")
with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED, allowZip64=True) as zipf:
    for folder in ["cache/siglip_features", "cache/siglip_meta"]:
        for root, _, files in os.walk(folder):
            for f in files:
                full_p = os.path.join(root, f)
                zipf.write(full_p, arcname=full_p)
    for root_file in ["cache/faiss_siglip.index", "cache/faiss_siglip_meta.pkl", "cache/features_matrix.npy"]:
        if os.path.exists(root_file):
            zipf.write(root_file, arcname=root_file)

size_mb = os.path.getsize(zip_filename) / (1024 * 1024)
print(f"[✓] Successfully created {zip_filename} ({size_mb:.2f} MB)!")
display(FileLink(zip_filename))